# Data Preparation

This notebook contains all steps and decisions made in the data preparation phase for the Austin Crime project.

## The Required Imports

Here we'll import all the required modules for this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from acquire import get_crime_data
import prepare

## Acquire the Data

We'll acquire the data using the get_crime_data function from the acquire module. Here we'll explicitly read from the source using an API, but going forward we will use the cache file 'Crime_Reports.csv'.

In [ ]:
# Acquire the data using the API

df = get_crime_data()
df.shape

In [ ]:
df.info()

## Limit Time Frame of the Data

We are only interested in crimes reported between the years 2018 and 2021. Here we'll remove all observations that occur outside of this time frame.

In [ ]:
# Let's see how the date information is stored in the dataframe.

df.head(1).occ_date

In [ ]:
# Set the occ_date column to a datetime type.

df.occ_date = pd.to_datetime(df.occ_date, format = '%Y-%m-%d')

In [ ]:
df.occ_date.head()

In [ ]:
# Subset the data to include observations between 2018-01-01 and 2021-12-31.

df = df[(df.occ_date >= '2018-01-01') & (df.occ_date <= '2021-12-31')]
df.shape

## Missing Values

Let's now investigate the missing values in our dataset and determine the best course of action for handling them.

### Summarize Null Values

In [ ]:
prepare.attribute_nulls(df)

### ucr_category and category_description

The ucr_category and category_description columns have the most missing values. Let's investigate these columns.

In [ ]:
df[['crime_type', 'ucr_code', 'ucr_category', 'category_description']].head()

The ucr_category and category_description columns have far too many missing values to be useful to us. Additionally, the crime_type column provides similar information so we will drop these two columns (see the Drop Columns section).

#### Attempting to Impute category_description

It may be possible to impute the category_description column, but because it's missing so many values it could be time consuming. Here we'll attempt to find an easy solution to this problem.

In [ ]:
# Look at all unique categories.

df.category_description.value_counts()

In [ ]:
# Gather a collection of all the crime types that are included in the existing category descriptions.

types = set()

for category in df.category_description.unique():
    types = types.union(df[df.category_description == category].crime_type.unique())

In [ ]:
len(types)

In [ ]:
len(set(df.crime_type.unique()).difference(types))

There are a large number of crime types that are not included in the existing category descriptions. Let's begin looking into these to see if we might be able to easily assign these to broader descriptions.

In [ ]:
unincluded_types = set(df.crime_type.unique()).difference(types)

In [ ]:
unincluded_types

Some of these would be easy to impute into a category, but there are far too many crime types to research here. Even using a shortcut like putting those that don't easily fit into a broad category into OTHER it would still be a very time consuming task to impute these values. It is definitely possible, but given the timeframe of this project it's not worth it to pursue this task.

### clearance_status

In [ ]:
# The target variable is missing some values as well. Let's investigate.

df.clearance_status.value_counts(dropna = False)

The values N, C, and O (according to the data documentation) mean a case is either closed or not closed. We cannot make a reasonable assumption for what the null values in this column might mean, but we cannot drop this column because this is our target variable. We will drop the rows missing this feature because this feature is critical to our project (see the Drop Rows section).

### computed_region columns

In [ ]:
df[[
    ':@computed_region_a3it_2a2z',
    ':@computed_region_8spj_utxs',
    ':@computed_region_q9nd_rr82',
    ':@computed_region_qwte_z96m'
]].head()

We have no idea what these columns might be, we'll drop them.

### clearance_date

In [ ]:
df.clearance_date.head()

This feature might be useful to us later on. It is missing roughly the same number of observations as the clearance_status column. We will drop all rows missing this column.

### location data

In [ ]:
df[[
    'x_coordinate',
    'y_coordinate',
    'latitude',
    'longitude',
    'location',
    'address'
]].head()

In [ ]:
# Let's see an observation of the location feature.

list(df.location.head(1))

The location feature is mostly the latitude and longitude repeated. We can drop this column. The x and y coordinate columns are likely relevant to the authors of the dataset and aren't very useful to us. We can drop these columns.

### location_type

In [ ]:
df.location_type.value_counts()

There aren't too many missing values in this column. There is also a value for OTHER/UNKNOWN that we can use to impute the missing values.

### zip_code and council_district

In [ ]:
df[['zip_code', 'council_district']].head(20)

In [ ]:
df.council_district.value_counts()

In [ ]:
df.zip_code.value_counts()

One of our initial questions depends on the council_district feature so we can't drop this column. It is possible that rows missing zip_code are also missing council_district. We will drop rows missing zip_code and then impute the remaining missing values in council_district.

### sector, district, and pra

We want to keep the sector and district columns as these may be useful to us in our exploration. We will remove the rows missing these values. The police reporting area column on the other hand may not be useful to us so we'll drop this column.

### census_tract

In [ ]:
df.census_tract.value_counts()

We don't think this column will be useful to us. We'll drop it.

## Drop Columns

Here we'll drop all columns that are either not useful or have too many missing values to be of any use to us.

In [ ]:
# These are all the columns that will be dropped from the dataframe.

columns = [
    'incident_report_number',
    'ucr_code',
    'ucr_category',
    'category_description',
    ':@computed_region_a3it_2a2z',
    ':@computed_region_8spj_utxs',
    ':@computed_region_q9nd_rr82',
    ':@computed_region_qwte_z96m',
    'x_coordinate',
    'y_coordinate',
    'location',
    'census_tract',
    'pra',
    'occ_date_time',
    'rep_date_time'
]

df = df.drop(columns = columns)
df.shape

## Drop Rows

Here we'll drop rows with missing values that cannot be reasonabled imputed with a value.

In [ ]:
df.clearance_status.value_counts(dropna = False)

In [ ]:
columns = [
    'clearance_status',
    'clearance_date',
    'zip_code',
    'sector',
    'district',
    'latitude',
    'longitude'
]

for column in columns:
    df = df[~df[column].isna()]

In [ ]:
df.shape

In [ ]:
prepare.attribute_nulls(df)

## Impute Missing Values

Here we'll fill missing values for some columns with a value we have decided upon.

In [ ]:
df['location_type'] = df.location_type.fillna('OTHER / UNKNOWN')
df['council_district'] = df.council_district.fillna(9)

In [ ]:
prepare.attribute_nulls(df)

## Rename Columns

Now for readability we will rename some of the columns to more easily understandable names.

In [ ]:
mapper = {
    'occ_date' : 'occurence_date',
    'occ_time' : 'occurence_time',
    'rep_date' : 'report_date',
    'rep_time' : 'report_time'
}

df = df.rename(columns = mapper)
df.info()

## Rename clearance_status Values

The values in the clearance_status column are rather unreadable. We will change the values to more readable values.

In [ ]:
# We'll use this map to rename the values in the clearance_status column.

mapper = {
    'N' : 'not cleared',
    'O' : 'cleared by exception',
    'C' : 'cleared by arrest'
}

df['clearance_status'] = df.clearance_status.map(mapper)

In [ ]:
df.clearance_status.value_counts()

## Assure Data Types Are Correct

Finally, let's ensure that the data types for all our columns are correct.

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# latitude and longitude are more accurately represented as numerical types.

df.latitude = df.latitude.astype('float')
df.longitude = df.longitude.astype('float')

In [ ]:
df.info()

In [ ]:
df.occurence_time = df.occurence_time.apply(lambda time: f'{int(time):04d}')
df.report_time = df.report_time.apply(lambda time: f'{int(time):04d}')

In [ ]:
# We want to change the date and time columns to datetime types.

df.report_date = pd.to_datetime(df.report_date, format = '%Y-%m-%d')
df.clearance_date = pd.to_datetime(df.clearance_date, format = '%Y-%m-%d')
df.occurence_time = pd.to_datetime(df.occurence_time, format = '%H%M')
df.report_time = pd.to_datetime(df.report_time, format = '%H%M')

df.occurence_time = df.occurence_time.dt.strftime('%H:%M')
df.report_time = df.report_time.dt.strftime('%H:%M')

df.info()

In [ ]:
df.head()

## Combine cleared_by_arrest and cleared_by_exception

The clearance_status column has three unique values: not cleared, cleared by arrest, and cleared by exception. We want to make this column binary so we will combine the cleared by arrest and cleared by exception values.

In [ ]:
df.clearance_status.value_counts()

In [ ]:
clearance = np.where(df.clearance_status == 'not cleared', False, True)
df['cleared'] = clearance
df.cleared.value_counts()

## Engineer Lockdown in Effect Feature

We are interested in seeing how pandemic lockdowns affected Austin crime. We will create a feature which indicates whether a crime occurred during a time when pandemic stay at home orders were in place.

In [ ]:
# Stay at home orders in Travis county began March 14, 2020. The end date for stay at home orders is tricky
# because stay at home orders were gradually lifted. We are using August 26, 2020 as an unofficial end to 
# stay at home orders. This is the date when the UT campus reopened.

df['pandemic_lockdown'] = np.where(((df.occurence_date >= '2020-03-14') & (df.occurence_date <= '2020-08-26')), True, False)